# Day 7 - Code breaks, block 1: autoencoders

Short code breaks that go with the slides (3-5 minutes each):
1. **run** the cell and look at the output,
2. **change** the value marked with `# 👈`, run again,
3. open **👀 What you should see** to check.

A **CPU runtime is enough** - every training takes 10-30 seconds. You cannot break anything - if something looks
strange: *Runtime → Restart session*, then run the first two cells again.

Data: **FashionMNIST** - 70 000 small grey pictures of clothes (28 × 28 = 784 pixels), 10 classes.
Today we use **no labels for training** - only to colour the pictures at the end.

## Setup - run these two cells first

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision.utils import make_grid

train_set = datasets.FashionMNIST("data", train=True, download=True)
test_set = datasets.FashionMNIST("data", train=False, download=True)
X = train_set.data.float().view(-1, 784) / 255          # 60 000 training pictures, 784 pixels each, values 0 ... 1
X_test = test_set.data.float().view(-1, 784) / 255      # 10 000 test pictures
y_test = test_set.targets                                # labels - only used for colours, never for training
print("training pictures:", tuple(X.shape), "  test pictures:", tuple(X_test.shape))

The second cell defines small helpers. **Just run it - you do not need to read it.**
- `make_autoencoder(bottleneck)` - an encoder 784 → 256 → 64 → `bottleneck` and a decoder `bottleneck` → 64 → 256 → 784
- `train(models, loss_fn, data, epochs)` - the training loop from day 3 (Adam, batches of 256), prints the loss
- `show_rows({"name": pictures, ...})` - draws one row of pictures per name
- `plot_map(points, labels)` - draws the points of a 2-number latent space, one colour per class
- `ONE` - the positions of one test picture per class (T-shirt, trousers, ..., boot)

In [ ]:
NAMES = ["T-shirt", "trousers", "pullover", "dress", "coat", "sandal", "shirt", "sneaker", "bag", "boot"]
ONE = torch.stack([(y_test == c).nonzero()[0, 0] for c in range(10)])


def make_autoencoder(bottleneck, n_in=784, last=nn.Sigmoid()):
    torch.manual_seed(0)
    encoder = nn.Sequential(nn.Linear(n_in, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, bottleneck))
    decoder = nn.Sequential(nn.Linear(bottleneck, 64), nn.ReLU(), nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, n_in), last)
    return encoder, decoder


def train(models, loss_fn, data, epochs, batch_size=256):
    params = [p for m in models for p in m.parameters()]
    opt = torch.optim.Adam(params, lr=1e-3)
    for epoch in range(epochs):
        total = 0.0
        for idx in torch.randperm(len(data)).split(batch_size):
            loss = loss_fn(data[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        print(f"epoch {epoch + 1}: loss {total / len(data):.4f}")


def show_rows(rows):
    fig, axes = plt.subplots(len(rows), 1, figsize=(10, 1.15 * len(rows)), squeeze=False)
    for ax, (name, imgs) in zip(axes[:, 0], rows.items()):
        grid = make_grid(imgs.detach().view(-1, 1, 28, 28), nrow=len(imgs), padding=2, pad_value=1)[0]
        ax.imshow(grid, cmap="gray", vmin=0, vmax=1); ax.set_xticks([]); ax.set_yticks([])
        for side in ax.spines.values(): side.set_visible(False)
        ax.set_ylabel(name, rotation=0, ha="right", va="center")
    plt.show()


def plot_map(points, labels, highlight=None):
    plt.figure(figsize=(8, 6))
    for c in range(10):
        on = highlight is None or NAMES[c] in highlight
        plt.scatter(*points[labels == c].T, s=2, color=plt.cm.tab10(c) if on else "#dddddd", label=NAMES[c])
    plt.legend(markerscale=6, loc="center left", bbox_to_anchor=(1, 0.5))
    plt.xlabel("latent dimension 1"); plt.ylabel("latent dimension 2")
    plt.show()


print("helpers ready")

## 💻 Code break 1 - Squeeze it through the bottleneck
The encoder squeezes 784 pixels into a few numbers (a point in the **latent space**), the decoder rebuilds the picture from them.
The loss compares the copy with the original - **the picture is its own label**.

In [ ]:
BOTTLENECK = 32      # 👈 try 2 and 8

encoder, decoder = make_autoencoder(BOTTLENECK)

def loss_fn(x):
    x_hat = decoder(encoder(x))          # compress, then rebuild
    return ((x_hat - x) ** 2).mean()     # MSE: how different is the copy from the original?

train([encoder, decoder], loss_fn, X, epochs=5)

with torch.no_grad():
    rebuilt = decoder(encoder(X_test))
print(f"test error (MSE): {((rebuilt - X_test) ** 2).mean():.4f}")
show_rows({"original": X_test[ONE], f"latent space: {BOTTLENECK} numbers": rebuilt[ONE]})

🔧 **Change it:** `BOTTLENECK = 2`, then `BOTTLENECK = 8`. Which details get lost first?

<details><summary>👀 What you should see</summary>

- test error after 5 epochs: **2 numbers 0.029**, **8 numbers 0.018**, **32 numbers 0.017**
- with 32 numbers the copies are good but **blurry** - fine patterns (stripes, logos, laces) are lost first
- with 2 numbers only the **rough shape** survives: some pictures turn into a different kind of clothing
  (a shirt becomes a pullover)
- 784 → 32 numbers is a compression of about 25 × - and nobody gave the network a label
</details>

## 💻 Code break 2 - The map: what did the 2 numbers learn?
With a latent space of only **2 numbers** we can draw every picture as a point. The colours are the classes - the network
never saw them.

In [ ]:
HIGHLIGHT = None     # 👈 e.g. ["sandal", "sneaker", "boot"] or ["T-shirt", "shirt", "pullover", "coat"]

encoder2, decoder2 = make_autoencoder(2)
train([encoder2, decoder2], lambda x: ((decoder2(encoder2(x)) - x) ** 2).mean(), X, epochs=5)

with torch.no_grad():
    latent = encoder2(X_test)            # 10 000 pictures → 10 000 points in the latent space
plot_map(latent, y_test, HIGHLIGHT)

🔧 **Change it:** highlight the shoes, then the tops. Which classes are neighbours, which ones overlap?

<details><summary>👀 What you should see</summary>

- pictures of the same class land **close together** - without any label: the autoencoder sorted the clothes by
  itself
- the shoes (sandal, sneaker, boot) are **neighbours**; trousers and bags have their own regions
- T-shirt, shirt, pullover and coat **overlap** - with 2 numbers they look too similar
- there are **gaps** between the clouds and the map has a strange shape - remember this for the VAE in block 2
</details>

## 💻 Code break 3 - The denoising autoencoder
Same network - only the training changes: the **input** gets noise, the **target** stays the clean picture.

In [ ]:
NOISE = 0.5          # 👈 try 0.2 and 0.8

def add_noise(x):
    return (x + NOISE * torch.randn_like(x)).clamp(0, 1)

encoder3, decoder3 = make_autoencoder(32)

def dae_loss(x):
    x_hat = decoder3(encoder3(add_noise(x)))   # input: the NOISY picture
    return ((x_hat - x) ** 2).mean()           # target: the CLEAN picture

train([encoder3, decoder3], dae_loss, X, epochs=5)

with torch.no_grad():
    torch.manual_seed(1)
    noisy = add_noise(X_test[ONE])
    show_rows({"original": X_test[ONE], "noisy input": noisy,
               "normal AE (break 1)": decoder(encoder(noisy)), "denoising AE": decoder3(encoder3(noisy))})

🔧 **Change it:** `NOISE = 0.8` - can you still recognise the clothes in the noisy row? Does the network?

<details><summary>👀 What you should see</summary>

- with `NOISE = 0.5` the denoising AE gives **clean** clothes back; the normal AE from code break 1 never saw noise
  and gives grey, messy copies
- measured on all 10 000 test pictures: noisy input error **0.118**, normal AE **0.056**, denoising AE **0.023**
- with `NOISE = 0.8` you can hardly see the clothes, but the network still finds the rough shape - it has learned
  what clothes **usually look like**
- the same idea removes eye blinks and muscle artefacts from EEG - and, repeated many times, it is the heart of
  diffusion models (Stable Diffusion)
</details>

## ⭐ Extra (if you are fast): find abnormal heartbeats
The ECG5000 heartbeats from day 5. We train the autoencoder **only on normal beats**. It learns to rebuild normal beats
well - an abnormal beat is rebuilt badly → **a large error = an alarm**.

In [ ]:
import pathlib, urllib.request, zipfile

root = pathlib.Path("data/ecg5000")
if not (root / "ECG5000_TEST.txt").exists():
    root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://www.timeseriesclassification.com/aeon-toolkit/ECG5000.zip", root / "ECG5000.zip")
    zipfile.ZipFile(root / "ECG5000.zip").extractall(root)
rows = [[float(v) for v in line.split()] for name in ("ECG5000_TRAIN.txt", "ECG5000_TEST.txt")
        for line in open(root / name)]
beats, kind = torch.tensor(rows)[:, 1:], torch.tensor(rows)[:, 0].long()      # 5 000 beats × 140 time steps; kind 1 = normal
g = torch.Generator().manual_seed(0)
normal = (kind == 1).nonzero()[:, 0]
normal = normal[torch.randperm(len(normal), generator=g)]
train_beats, test_normal, test_abnormal = beats[normal[:2000]], beats[normal[2000:]], beats[kind != 1]
print("training (normal only):", len(train_beats), "  test normal:", len(test_normal), "  test abnormal:", len(test_abnormal))

In [ ]:
ALARM_LEVEL = 0.95   # 👈 the threshold lets this share of the normal TRAINING beats pass - try 0.99

enc_ecg, dec_ecg = make_autoencoder(8, n_in=140, last=nn.Identity())
train([enc_ecg, dec_ecg], lambda x: ((dec_ecg(enc_ecg(x)) - x) ** 2).mean(), train_beats, epochs=30, batch_size=64)

with torch.no_grad():
    error = lambda b: ((dec_ecg(enc_ecg(b)) - b) ** 2).mean(dim=1)      # one error per beat
    threshold = torch.quantile(error(train_beats), ALARM_LEVEL)
    print(f"threshold {threshold:.3f}")
    print(f"alarm for normal beats:   {(error(test_normal) > threshold).float().mean():.0%}   (false alarms)")
    print(f"alarm for abnormal beats: {(error(test_abnormal) > threshold).float().mean():.0%}   (found)")
    plt.hist(error(test_normal).clamp(max=0.6), 50, alpha=0.7, label="normal", color="green")
    plt.hist(error(test_abnormal).clamp(max=0.6), 50, alpha=0.6, label="abnormal", color="red")
    plt.axvline(threshold, color="black", ls="--"); plt.xlabel("reconstruction error"); plt.legend(); plt.show()

🔧 **Change it:** `ALARM_LEVEL = 0.99` - fewer false alarms, but do you miss abnormal beats?

<details><summary>👀 What you should see</summary>

- normal beats have a small error (median ≈ 0.01), abnormal beats a large one (median ≈ 0.17)
- `ALARM_LEVEL = 0.95`: threshold ≈ 0.034 → **8 %** false alarms on new normal beats, **100 %** of the abnormal
  beats found
- `ALARM_LEVEL = 0.99`: threshold ≈ 0.094 → only **1 %** false alarms, but **94 %** found - 6 % of the abnormal
  beats are missed
- a higher threshold → fewer false alarms, more missed beats: the trade-off from day 3 (sensitivity vs specificity)
- this only needs **normal** data for training - useful when the abnormal class is rare (seizures!)
</details>